# Clase 3 — ¿De dónde saco los datos?

**Sistemas de Información Geográfica**
Especialización en Ciencias Sociales Computacionales — Universidad Nacional Guillermo Brown

| | |
|---|---|
| **Unidad del programa** | 3 — Fuentes de información geográfica |
| **Duración** | 3 horas |
| **Versión** | 2026.1 |
| **Docente** | Renzo Polo |
| **Licencia** | CC BY-SA 4.0 |

---

## 1. La pregunta de hoy

> ### ¿De dónde saco datos para mi propia pregunta de investigación?

Hoy vamos a buscar datos, de tres grandes fuentes: **portales de datos abiertos**, **OpenStreetMap**
y **geoservicios de una IDE**.

## 2. Objetivos de esta clase

Al terminar, deberías poder:

1. **Descargar** una tabla de un portal de datos abiertos y **convertirla** en una capa
   espacial a partir de su columna de geometría en formato WKT.
2. **Consultar** un geoservicio WFS: escribir su dirección, pedirle el catálogo de capas
   que publica y descargar una de ellas, y **distinguirlo** de un WMS.
3. **Obtener** elementos de OpenStreetMap por etiqueta y por lugar, y **medir** que su
   cobertura no es uniforme en el territorio.
4. **Detectar** cuándo una capa llegó sin sistema de coordenadas declarado, y explicar por
   qué eso invalida cualquier medición aunque el mapa se vea bien.

## 3. Material de esta clase

- **Presentación Clase 3**, diapositivas 1–20.

| Bloque de la notebook | Diapositivas |
|---|---|
| Portales de datos abiertos | 3–6 |
| OpenStreetMap | 7–9 |
| IDE, geoservicios, WMS vs. WFS | 10–20 |

Direcciones que vamos a usar:

| Recurso | Dirección |
|---|---|
| Poblaciones (Censo 2022) | <https://poblaciones.org> |
| Etiquetas de OpenStreetMap | <https://wiki.openstreetmap.org/wiki/ES:Objetos_del_mapa> |
| Geoservicios del IGN | <https://www.ign.gob.ar/AreaServicios/Geoservicios> |
| Portal nacional de datos abiertos | <https://datos.gob.ar> |

## 4. Preparación del entorno

In [ ]:
!pip install -q "geopandas==1.0.1" "folium==0.17.0" "matplotlib==3.9.2" "osmnx==2.0.5" "OWSLib==0.31.0"
!wget -q -O sig_utils.py https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/sig_utils.py

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from sig_utils import chequear_crs, CRS_ARGENTINA

# Los datos del curso viven en un repositorio público de GitHub.
DATOS = "https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/datos/"

# Un sistema que conserva las superficies, para cuando midamos áreas.
EQUIVALENTE = CRS_ARGENTINA["sudamerica_equivalente"]

provincias = gpd.read_file(DATOS + "provincias_arg.gpkg")
print(f"GeoPandas {gpd.__version__} — entorno listo")

---

## 5. Un portal de datos abiertos

> 📽️ **Presentación Clase 3, diapositivas 3–6** — Portales de datos abiertos y licencias.

Un portal de datos abiertos publica información del Estado bajo una licencia que permite
reutilizarla. La mayoría entrega **tablas**, no capas: un CSV con una columna de más, donde
la geometría viene escrita como texto en formato **WKT** (*Well-Known Text*), así:

```
POLYGON((-58.386 -34.578, -58.375 -34.580, ... ))
```

Vamos a trabajar con la descarga de [Poblaciones](https://poblaciones.org), que publica los
datos del Censo 2022 por departamento. **El archivo está tal como sale del portal**: no le
corregimos nada.

In [ ]:
# El CSV que se baja del portal. Lo abrimos con pandas, como cualquier tabla.
censo = pd.read_csv(DATOS + "poblaciones_departamentos_2022.csv")

print(f"{len(censo)} filas y {len(censo.columns)} columnas")
censo.head()

### ▶️ De columna de texto a capa espacial

`GeoSeries.from_wkt()` lee ese texto y lo convierte en geometrías de verdad. Con eso ya
podemos armar un GeoDataFrame.

In [ ]:
departamentos = gpd.GeoDataFrame(
    censo,
    geometry=gpd.GeoSeries.from_wkt(censo["Geometría en WKT"]),
)

print("Sistema de coordenadas:", departamentos.crs)

### 👀 Y sin embargo, el mapa sale bien

In [ ]:
fig, eje = plt.subplots(figsize=(6, 9))
departamentos.plot(ax=eje, facecolor="#dce8f2", edgecolor="white", linewidth=0.3)
provincias.boundary.plot(ax=eje, color="grey", linewidth=0.8)
eje.set_title("527 departamentos, dibujados desde el WKT")
eje.set_axis_off()
plt.show()

### ✅ Comprobación — midamos algo

El mapa se ve como esperábamos, así que la tentación es seguir adelante. Antes, una
comprobación barata: el propio CSV trae la superficie oficial de cada departamento en km².
Calculemos nosotros esa superficie y comparemos.

In [ ]:
departamentos["superficie_calculada"] = departamentos.geometry.area

comparacion = departamentos[["Nombre de departamentos/comuna",
                             "Superficie en km2", "superficie_calculada"]]
print(comparacion.head(5).to_string(index=False))

### 🔍 El número salió, y está mal

No hubo ningún error ni ninguna advertencia: GeoPandas midió y devolvió un número. Pero ese
número no son kilómetros cuadrados. Son **grados cuadrados**, que no significan nada: un
grado mide distinto en Ushuaia que en La Quiaca.

Lo que pasa es que un CSV con números no es un dato espacial. Nadie declaró qué son esas
coordenadas, y GeoPandas no lo puede adivinar: por eso `departamentos.crs` imprimió `None`.
Y cuando el sistema no está declarado, tampoco se lo puede convertir a otro:

### ▶️ La solución: declararlo, con el dato de la fuente

`set_crs()` **declara** lo que las coordenadas ya son. No mueve nada: le pone nombre a lo
que estaba. El valor no se inventa, sale de la documentación del portal, que publica en
coordenadas geográficas WGS 84, es decir **EPSG:4326**.

Recién con el sistema declarado se puede convertir a otro para medir.

In [ ]:
departamentos = departamentos.set_crs("EPSG:4326")

en_metros = departamentos.to_crs(EQUIVALENTE)
departamentos["superficie_calculada"] = en_metros.geometry.area / 1_000_000  # m² a km²

comparacion = departamentos[["Nombre de departamentos/comuna",
                             "Superficie en km2", "superficie_calculada"]]
print(comparacion.head(5).to_string(index=False))

### 🔍 Interpretación

Las dos columnas ahora coinciden. El dato siempre estuvo bien: lo que faltaba era el
**metadato** que dice qué significan esos números.

Y queda una pregunta abierta, que es la de la clase que viene: **¿por qué EPSG:4326 y no
otro? ¿Y por qué para medir hubo que pasar a un tercero?**

---

## 6. OpenStreetMap

> 📽️ **Presentación Clase 3, diapositivas 7–9** — OSM y el mapeo colaborativo.

OpenStreetMap es un mapa del mundo hecho por voluntarios. Todo lo que hay adentro está
descrito con **etiquetas**, pares de `clave = valor`: un bar es `amenity = bar`, una escuela
es `amenity = school`, un kiosco es `shop = kiosk`. El listado completo está en el
[wiki de OSM](https://wiki.openstreetmap.org/wiki/ES:Objetos_del_mapa).

La biblioteca **OSMnx** consulta OSM desde Python. `features_from_place()` necesita dos
cosas: el nombre de un lugar y las etiquetas que queremos.

In [ ]:
import osmnx as ox

etiqueta = {"amenity": True}   # True = cualquier valor de amenity

recoleta = ox.features_from_place(
    "Recoleta, Ciudad Autónoma de Buenos Aires, Argentina", tags=etiqueta)

print(f"{len(recoleta)} elementos con etiqueta amenity en Recoleta")
print("Sistema de coordenadas:", recoleta.crs)
recoleta["amenity"].value_counts().head(8)

### 👀 Dónde están

In [ ]:
limite = ox.geocode_to_gdf("Recoleta, Ciudad Autónoma de Buenos Aires, Argentina")

fig, eje = plt.subplots(figsize=(7, 7))
limite.boundary.plot(ax=eje, color="grey")
recoleta.plot(ax=eje, color="#c0392b", markersize=6)
eje.set_title("Elementos con etiqueta amenity — Recoleta")
eje.set_axis_off()
plt.show()

### ▶️ El mismo pedido, en otro barrio

Repetimos la consulta en Villa Lugano, en el otro extremo de la misma ciudad. Como los
barrios no tienen la misma superficie, comparamos **densidad**: elementos por km².

In [ ]:
barrios = ["Recoleta, Ciudad Autónoma de Buenos Aires, Argentina",
           "Villa Lugano, Ciudad Autónoma de Buenos Aires, Argentina"]

for lugar in barrios:
    elementos = ox.features_from_place(lugar, tags=etiqueta)
    contorno = ox.geocode_to_gdf(lugar).to_crs(EQUIVALENTE)
    km2 = contorno.geometry.area.iloc[0] / 1_000_000

    nombre = lugar.split(",")[0]
    print(f"{nombre:15} {len(elementos):5} elementos  {km2:6.1f} km²  "
          f"{len(elementos) / km2:6.1f} por km²")

### 🔍 Lo colaborativo no es uniforme

Para un científico social esto no es un detalle técnico, es el problema de fondo de la
fuente. Un mapa colaborativo se llena donde hay gente con tiempo, conexión y costumbre de
mapear. Si usáramos OSM como si fuera un censo de equipamiento urbano, concluiríamos que
los barrios de menores ingresos tienen menos servicios, cuando parte de esa diferencia es
**de la fuente y no del territorio**. Toda conclusión sacada de OSM tiene que declarar esto.

Una diferencia con el bloque anterior: acá el CRS **sí** llegó declarado —OSM publica en
EPSG:4326 y lo dice—, así que la reproyección para calcular los km² funcionó sin que
tuviéramos que declarar nada. La diferencia no está en el dato: está en quién lo publica y
en si se tomó el trabajo de decir en qué sistema está.

---

## 7. Un geoservicio: el WFS del IGN

> 📽️ **Presentación Clase 3, diapositivas 10–20** — IDE, geoservicios, WMS vs. WFS.

Una **IDE** (Infraestructura de Datos Espaciales) publica los datos de un organismo a
través de servicios estandarizados, que se consultan por su dirección web:

| Servicio | Qué devuelve | Para qué sirve |
|---|---|---|
| **WMS** | Una **imagen** del mapa ya dibujado | Mirar. No se puede consultar la tabla ni medir. |
| **WFS** | Las **geometrías y sus atributos** | Trabajar: filtrar, medir, cruzar con otras capas. |

Hoy usamos el WFS del Instituto Geográfico Nacional. La dirección se saca de la [página de
geoservicios del organismo](https://www.idera.gob.ar/index.php/servicios/geoservicios), y siempre termina en `/wfs` o `/ows`.

Lo primero que se le pregunta a un servicio es **qué tiene**: eso es la operación
`GetCapabilities`, que OWSLib hace sola al conectarse.

In [ ]:
from owslib.wfs import WebFeatureService

wfs_url = "https://wms.ign.gob.ar/geoserver/ows"

wfs = WebFeatureService(wfs_url, version="2.0.0")

print("Servicio:", wfs.identification.title)
print("Capas publicadas:", len(wfs.contents))

### ▶️ Qué publica el servicio

`wfs.contents` es el catálogo: un diccionario donde cada capa trae su nombre técnico, su
título y un resumen. Lo recorremos entero para ver con qué nos encontramos. Son 192, así
que la salida es larga: conviene recorrerla por arriba, sin leer una por una.

In [ ]:
# Listado de capas publicadas
for layer_name, layer in wfs.contents.items():
    print(f"Capa: {layer_name}")
    print(f"Título: {layer.title}")
    print(f"Resumen: {layer.abstract}")
    print("-" * 40)

### ▶️ Buscar una capa en el catálogo

Leerlas de a una no es viable. Recorremos el mismo diccionario y nos quedamos solo con las
que mencionan una palabra en su nombre. El nombre técnico es críptico; el título dice qué
hay adentro.

In [ ]:
palabra = "salud"

for nombre, capa in wfs.contents.items():
    if palabra in nombre:
        print(f"{nombre:35} {capa.title}")

### ▶️ Descargar una capa

`getfeature()` pide los datos de la capa que elegimos, y GeoPandas lee la respuesta igual
que si fuera un archivo.

In [ ]:
wfs = WebFeatureService(wfs_url, version="2.0.0")

respuesta = wfs.getfeature(typename=["ign:salud_020801"])
salud = gpd.read_file(respuesta)

print(f"{len(salud)} establecimientos")
chequear_crs(salud, proyectado=False, nombre="edificios de salud (IGN)")
salud[["fna", "nam", "geometry"]].head()

### 🔍 Acá el sistema vino con los datos

No tuvimos que declarar nada. La capa llegó con su sistema de coordenadas puesto, porque el
servicio lo manda **junto con las geometrías**, en la misma respuesta: el estándar obliga a
que cada conjunto de coordenadas diga en qué sistema está.

Es lo contrario de lo que nos pasó con el CSV del portal, que traía los números sueltos. Un
geoservicio no te entrega solo el dato: te entrega el dato **y su metadato**, y por eso se
puede combinar con otras capas sin averiguar nada por fuera.

### 👀 La capa que bajamos, sobre el país

In [ ]:
fig, eje = plt.subplots(figsize=(6, 9))
provincias.boundary.plot(ax=eje, color="grey", linewidth=0.8)
salud.plot(ax=eje, color="#c0392b", markersize=1)
eje.set_title(f"{len(salud)} edificios de salud — WFS del IGN")
eje.set_axis_off()
plt.show()

### ▶️ Guardar lo descargado

Un servicio puede estar caído mañana, así que lo primero después de bajar algo es guardarlo.
Lo hacemos en **GeoPackage**: en la Clase 2 vimos qué le hace el Shapefile a los nombres de
las columnas y a los acentos.

In [ ]:
salud.to_file("salud_ign.gpkg")

control = gpd.read_file("salud_ign.gpkg")
print(f"Guardado y releído: {len(control)} filas, CRS {control.crs}")

---

## 8. Cierre

### Las tres fuentes, y el mismo problema

| Fuente | Cómo llega el dato | El sistema de coordenadas |
|---|---|---|
| Portal de datos abiertos | Un CSV con la geometría como texto | **No viene.** Hay que declararlo con `set_crs()` |
| OpenStreetMap | Una consulta por etiqueta | Viene declarado (EPSG:4326) |
| Geoservicio WFS | Una respuesta del servidor | **Viaja junto a las geometrías**, en la misma respuesta |

Las tres veces, antes de poder medir cualquier cosa, hubo que saber qué son esas
coordenadas. Las dos fuentes pensadas para publicar datos espaciales —OSM y el geoservicio—
lo mandan con el dato; el portal, que publica tablas, lo dejó afuera. Y **el mapa se veía
perfecto igual**: el dibujo no avisa nada, el error aparece recién cuando se mide.

### Glosario de la clase

| Término | Definición |
|---|---|
| **IDE** | Infraestructura de Datos Espaciales: el conjunto de datos, servicios y acuerdos con que un organismo publica su información geográfica. |
| **WMS** | Servicio que devuelve una imagen del mapa. Sirve para mirar. |
| **WFS** | Servicio que devuelve geometrías y atributos. Sirve para trabajar. |
| **GetCapabilities** | Operación que le pregunta a un servicio qué capas publica. |
| **GetFeature** | Operación que pide los datos de una capa. |
| **WKT** | *Well-Known Text*: la geometría escrita como texto, tal como la traen los CSV de los portales. |
| **Etiqueta de OSM** | Par `clave = valor` con que OpenStreetMap describe cada objeto. |
| **`set_crs()`** | Declara el sistema de coordenadas que la capa ya tenía. No mueve nada. |

### La próxima clase

Hoy tratamos el sistema de coordenadas como un trámite: si falta, se declara; si hay que
medir, se convierte. Pero no dijimos **qué es**. La Clase 4 responde las preguntas que
quedaron abiertas:

> ¿Por qué EPSG:4326 y no otro? ¿Por qué para medir superficies hubo que convertir a un
> tercer sistema? ¿Y qué pasa si uno mide en el equivocado?

## 9. Bibliografía

- Instituto Geográfico Nacional (2024). *Geoservicios del IGN*.
  <https://www.ign.gob.ar/AreaServicios/Geoservicios>
- Boeing, G. (2017). "OSMnx: New Methods for Acquiring, Constructing, Analyzing, and
  Visualizing Complex Street Networks". *Computers, Environment and Urban Systems*, 65,
  126–139. <https://doi.org/10.1016/j.compenvurbsys.2017.05.004>
- Mooney, P. y Minghini, M. (2017). "A Review of OpenStreetMap Data". En *Mapping and the
  Citizen Sensor* (pp. 37–59). Ubiquity Press. <https://doi.org/10.5334/bbf.c>
- Open Geospatial Consortium (2010). *OpenGIS Web Feature Service 2.0 Interface Standard*.
  <https://www.ogc.org/standards/wfs>